# Ablation experiments

Goal:
Compare how different feature and preprocessing choices affect model quality.

We will compare several variants:

1. all features with missing indicators
2. all features without missing indicators
3. without Insulin
4. without SkinThickness
5. without Insulin and SkinThickness
6. without Pregnancies

For each variant we will evaluate:

- Logistic Regression
- k-NN with n_neighbors = 15
- Decision Tree with max_depth = 4

All comparisons are performed using cross-validation on X_train only.

The final X_test set is not used for model selection.

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score

from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_utils import load_data, split_features_target, make_train_test_split

from src.preprocessing import (
    replace_suspicious_zeros,
    replace_suspicious_zeros_with_indicators,
)

In [5]:
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "diabetes.csv"

df = load_data(DATA_PATH)

X, y = split_features_target(df)

X_train, X_test, y_train, y_test = make_train_test_split(X, y)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True))

X_train shape: (614, 8)
X_test shape: (154, 8)

Train target distribution:
Outcome
0    0.651466
1    0.348534
Name: proportion, dtype: float64

Test target distribution:
Outcome
0    0.649351
1    0.350649
Name: proportion, dtype: float64


In [6]:
X_train_with_indicators = replace_suspicious_zeros_with_indicators(X_train)

print("Original X_train shape:", X_train.shape)
print("With indicators shape:", X_train_with_indicators.shape)
print(X_train_with_indicators.columns.tolist())
print(X_train_with_indicators[["Insulin_missing", "SkinThickness_missing"]].mean())

Original X_train shape: (614, 8)
With indicators shape: (614, 10)
['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Insulin_missing', 'SkinThickness_missing']
Insulin_missing          0.472313
SkinThickness_missing    0.285016
dtype: float64
